# Reporte de hallazgos — Réplica y actualización de Aradillas (2018)

**Insumo para white paper.** Documento de trabajo interno; los números provienen de
`aradillas_2014.ipynb`, `aradillas_2022.ipynb` y `comparacion_2014_2022.ipynb`.

*Última actualización: 15 de septiembre de 2026.*

---

## Resumen

Replicamos en Python el estudio de Aradillas López (2018) para COFECE y lo actualizamos con
la ENIGH 2022. El ejercicio produjo tres tipos de resultado:

1. **Una réplica que valida el estudio en lo esencial** — el Gini observado coincide al
   decimal (0.481), las elasticidades regionales caen todas dentro de ±0.15, y los
   parámetros de poder de mercado quedan en el rango publicado.

2. **Discrepancias verificables entre lo que el paper declara y lo que su código hace**,
   ocho en total. Una de ellas —la ausencia de controles de costo por sector— compromete
   la identificación del parámetro central del estudio.

3. **Una actualización a 2022** que muestra que el costo del poder de mercado para los
   hogares creció, con evidencia sólida en los parámetros de markup y evidencia preliminar
   en las magnitudes de bienestar.

Sobre el acceso al código original: `programa_ENIGH_2014.g` se obtuvo por solicitud de
acceso a la información pública. **Nada de la sección 2 de este reporte habría sido
detectable leyendo únicamente el documento publicado.**

---

## Cómo leer este reporte

| sección | contenido | uso sugerido en el white paper |
|---|---|---|
| 1 | Qué replica y qué no | validación / metodología |
| **2** | **Discrepancias código ↔ paper** | **contribución principal** |
| 3 | El hallazgo sobre escalabilidad del estimador | contribución metodológica |
| 4 | Resultados 2022 y comparación | resultados |
| 5 | Limitaciones | honestidad metodológica |
| 6 | Agenda abierta | trabajo futuro |

---

# 1. Qué replica el ejercicio y qué no

| resultado | réplica | paper | veredicto |
|---|---|---|---|
| **Gini observado** | **0.481** | **0.481** | exacto |
| Gini contrafactual | 0.451 | 0.446 | ±0.005 |
| Cuadro 5 — elasticidades por región | 8/8 dentro de ±0.15 | | replica |
| Cuadro 8 — β_η | rango publicado (0.02–1.48) | | replica en estructura |
| Cuadro 4 — elasticidades nacionales | MAE 0.239, 10/13 dentro de ±0.30 | | parcial |
| Muestra final | 8,940 hogares | 15,586 | **no reproducible** |

**La brecha muestral no se explica.** El paper declara 15,586 hogares (≈80 % de la ENIGH) y
ninguna combinación de los filtros documentados la reproduce. Nuestro universo es de 12,372
antes del recorte iterativo y 8,940 después. Ver §2, divergencias B y C.

### Doce errores de implementación corregidos

Durante la réplica y la actualización se identificaron y corrigieron doce errores **de nuestro propio código**
(no del estudio original), todos verificados contra el programa Gauss. Se listan porque
condicionan la lectura de versiones anteriores de este trabajo, incluidas las que Victor
produjo antes del refactor:

| # | error | efecto |
|---|---|---|
| N4 | columnas del archivo de precios leídas como contiguas | transporte aéreo a \$138 en vez de \$2,279 |
| **N8/N8b** | **pesos del índice Divisia sin sumar 1** | **elasticidades comprimidas a −1** |
| N10 | Gini sobre `ing_total` y muestra filtrada | Gini 0.448 en vez de 0.481 |
| N11 | denominador de la VE equivocado | VE/ingreso 10.0 % en vez de 15.8 % |
| N6, N12, N13, N14 | filtro de selección, `nanmean`, fallback a precios originales, solver | varios |
| **N15** (2022) | **controles de costo de los Censos Económicos asignados por posición** | **ninguna de las 46 ciudades recibía sus datos; 4 de 7 variables eran otras** |
| N16 (2022) | base del deflactor = mediana de 2018 en vez de julio 2018 | 22 % de los factores desviados >5 %, hasta ±50 % en frutas estacionales |
| — (2022) | Iguala ubicada en el municipio de Igualapa | 210 km de error en la asignación de hogares |

El detalle completo está en `CLAUDE.md`. **N8 es el relevante para la narrativa**: era la
causa de que las elasticidades salieran pegadas a −1 y los markups saturados, tanto en 2014
como en 2022. **N15 es el relevante para 2022**: con él, el β_η de Pan salía 1.475; sin él, 1.219 (ambos antes de incluir el total estatal de la CDMX, que lo lleva a 1.332).

---

# 2. Discrepancias entre el código y el documento publicado

**Ésta es la contribución principal del proyecto.** Ocho discrepancias verificadas, con su
ubicación exacta en el programa original.

## 2.1 Las cuatro que afectan la interpretación de los resultados

### D-H — Los controles de costo no son específicos por sector

> **El paper** (p. ~1377): *"variables de costos de insumos por empresa **específicos para
> aquellas ramas de actividad económica relacionadas con cada una de las doce categorías**
> de gasto"*. El **Cuadro 7** presenta el mapeo detallado a ramas SCIAN.

**El código** (l.6182–6209):

1. Carga `indicadores_costos_censos_economicos_2014.asc` como `[46,11]`: 46 filas = 46
   **ciudades**, 11 columnas de indicadores. **El archivo no tiene dimensión de sector.**
2. Las líneas 6197–6209 son **trece reasignaciones consecutivas e incondicionales** de
   `vars_costos`. Cada una sobrescribe la anterior: solo sobrevive la última. Las otras doce
   son código muerto.
3. El bloque está **dentro del bucle de categorías** (l.5931–6324) y se reejecuta idéntico
   en cada vuelta.

**Consecuencia:** tortillas, pan, carne de res, medicamentos y transporte aéreo comparten el
mismo vector de controles de costo, que son características de la **ciudad**, no del sector.

**Por qué compromete la conclusión central:** el modelo NEIO existe para separar la
variación de precios atribuible a costos de la atribuible a poder de mercado. Sin controles
de costo que varíen por sector, toda variación de precio específica de un sector se atribuye
a poder de mercado **por construcción**. El sesgo sobre `β_η` es al alza y sistemático.

### D-A — El estimador descrito no es el implementado

El paper describe una estimación en **dos etapas (OLS + GMM)**. El programa implementa
**únicamente el bucle OLS** de 16 iteraciones (l.2452–5753); no existe segunda etapa. Los
parámetros publicados son OLS.

### D-B — Un filtro de muestra no declarado

El código descarta los hogares que no son propietarios de su vivienda (l.1101), **el 27 % de
la muestra (5,215 hogares)**. El paper no lo menciona al describir el universo del estudio,
que declara construido solo con el criterio de distancia y el de relevancia de categorías.
El sesgo hacia propietarios no está discutido.

### D-D — El recorte iterativo no declarado

El programa recorta el 1 % de cada cola de la utilidad **en cada una de las 16 iteraciones**,
de forma acumulativa (l.5671–5722, dentro del bucle). Se pierde el **28 %** de la muestra:
12,372 → 8,940. El paper no lo menciona, y la muestra efectiva de estimación no es la que
reporta.

## 2.2 Las cuatro restantes

| # | discrepancia | ubicación |
|---|---|---|
| D-C | Los 15,586 hogares declarados no son reconstruibles con los criterios publicados | — |
| D-E | Un piso numérico (0.01) sobre gastos nulos contamina los pesos de los índices de precio: en transporte, el 71.5 % de los hogares recibe un reparto 50/50 inventado (real: 73/27); en carne de res, las vísceras pesan 22.8 % cuando su participación real es 3.4 % | l.1766 y análogas |
| D-F | La categoría "Pan" (claves A012 pan blanco y A013 pan dulce en piezas) **deja fuera el pan industrial**: A014 pan dulce empaquetado y A015 pan para sándwich. **El CD incluye `precios_pan_de_caja_*.asc` y el programa nunca los usa** | l.361-363, 1979-1990 |
| D-G | Tres correcciones verificadas **alejan** los resultados de las cifras publicadas | N4, N9, N13 |

### Sobre D-G

Corregir errores demostrables mueve el Gini contrafactual de 0.446 a 0.442, la VE/ingreso de
15.8 % a 17.4 % y el MAE de 0.218 a 0.251 — **en dirección contraria a lo publicado**. La
lectura natural es que las cifras del paper incorporan errores que se compensan entre sí.

Es un argumento sobre **robustez**, no sobre honestidad: no se puede hacer sin haber
replicado, y es la razón por la que el criterio de este proyecto es fidelidad al código y no
a los números publicados.

### Sobre D-F, y su relación con D-H

Pan es **sistemáticamente el sector con más poder de mercado medido**: β_η de 1.477 en el
paper (el 2º más alto), 1.011 en nuestra réplica 2014 y **1.373 en 2022** (el más alto,
t = 9.31).

La hipótesis inicial era que "Pan" mezclaba un mercado concentrado (industrial) con uno
atomizado (panadería). **El catálogo ENIGH lo descarta: el pan industrial nunca estuvo en la
categoría.** La omisión es lo contrario de una mezcla: el segmento concentrado **no se mide**.
Lo medimos en 2022 como sector propio (clave A015, genérico INPC "Pan de caja"):

| sector 2022 | β_η | t | elasticidad |
|---|---|---|---|
| Pan tradicional (A012 + A013) | 1.373 | 9.31 | 1.423 |
| **Pan de caja (A015)** | **0.971** | **10.01** | 1.237 |

Dos resultados: **el pan industrial tiene poder de mercado alto y propio** (el 2º sector), y
**el β_η de la panadería tradicional casi no cambia al separarlo** (1.332 sin la categoría,
1.373 con ella) — no era un promedio contaminado. Que el sector atomizado muestre el mayor poder de
mercado medido es contraintuitivo y apunta a D-H: sin controles de costo por sector, la
variación de precios del pan tradicional se atribuye a poder de mercado por construcción.

D-F es sobre la definición de la **categoría de gasto**; D-H es sobre el lado de los
**costos**. Son independientes y se acumulan.

---

# 3. Hallazgo metodológico: el estimador no escala

**El algoritmo del estudio original deja de funcionar cuando la encuesta crece.**

El recorte iterativo (D-D) es inocuo con la muestra de 2014 y destruye la identificación con
la de 2022. No es un error de programación: es una propiedad del estimador que **solo se
vuelve visible al actualizar el ejercicio con datos nuevos**.

| | 2014 (8,940 hog.) | 2022 (57,552 hog.) |
|---|---|---|
| varianza del regresor de utilidad | 1.170 | **0.450** |
| hogares con efectos ingreso planos | 6.9 % | **76.1 %** |
| convergencia del solver | ~97 % | **31.9 %** |
| rango de elasticidades | [0.69, 1.74] | **[0.04, 3.35]** |

**Mecanismo.** El recorte elimina las colas de la utilidad en cada iteración. Con 8,940
hogares esas colas se regeneran entre iteraciones y la varianza sobrevive; con 57,552 los
cuantiles son estables, el recorte muerde siempre en el mismo lugar y la varianza colapsa.
Sin varianza en el regresor, los coeficientes de efectos ingreso quedan sin identificar, la
función de costo se vuelve plana y el solver de utilidad no converge.

**Y el efecto es asimétrico:** desactivar el recorte *mejora* 2022 y *empeora* 2014
(MAE 0.217 → 0.286). El recorte no es un defecto en abstracto — está calibrado para un
tamaño de muestra concreto, sin que nada lo advierta.

**Implicación práctica:** cualquier actualización del estudio con ENIGH moderna (2022, 2024)
debe desactivar el recorte o recalibrarlo. Aplicarlo tal cual produce resultados
inservibles con apariencia de normalidad — que es exactamente lo que ocurrió en la primera
versión de nuestra actualización, con las 13 elasticidades pegadas a −1.

---

# 4. Resultados de la actualización 2022

*Cifras de `comparacion_2014_2022.ipynb` (re-ejecutado el 15 de septiembre de 2026, con las
correcciones N15, N16 e Iguala): ambos años sin recorte, V1–V4.*

## 4.1 Poder de mercado — el resultado sólido

| categoría | β 2014 (t) | β 2022 (t) | cambio | |
|---|---|---|---|---|
| **Materiales de construcción** | 0.009 (1.26) | **0.429 (3.99)** | **+0.420** | gana significancia |
| **Pan** | 1.011 (9.20) | **1.373 (9.31)** | **+0.361** | el más alto |
| Medicamentos | 0.310 (3.29) | 0.549 (9.45) | +0.239 | |
| Tortillas | 0.119 (2.15) | 0.326 (3.73) | +0.207 | gana significancia |
| Bebidas | 0.272 (3.71) | 0.324 (3.87) | +0.051 | |
| Carnes procesadas | 0.197 (1.52) | 0.207 (5.59) | +0.010 | gana significancia |
| Transporte foráneo | 0.032 (2.42) | 0.023 (0.42) | −0.009 | pierde significancia |
| Pollo y huevo | 0.268 (2.71) | 0.115 (2.17) | −0.153 | pierde significancia |
| Frutas | 0.966 (7.86) | 0.752 (7.30) | −0.214 | |
| Carne de res | 0.403 (4.16) | 0.158 (5.74) | −0.245 | |
| Verduras | 0.502 (7.77) | −0.006 (−0.12) | −0.508 | pierde significancia |
| Lácteos | 0.726 (5.53) | 0.090 (1.54) | −0.636 | pierde significancia |
| **Pan de caja** | — | **0.971 (10.01)** | *nuevo* | solo 2022 (ver §2, D-F) |

*(Significancia: t ≥ 2.326 y β > 0, V1. 9 sectores significativos en 2014 y 9 en 2022, de
12 y 13 respectivamente.)*

**Lectura:** el poder de mercado se **reconfigura** más que crecer parejo. Gana en
**materiales de construcción, pan, medicamentos y tortillas** —vivienda, alimento básico y
salud—, aparece un sector nuevo muy concentrado (**pan de caja**), y retrocede en lácteos,
verduras, carne de res, frutas y pollo.

**Robustez.** Estas cifras sustituyen a las de la versión del 5 de agosto (Pan 1.496,
Materiales 0.515), que tenían los controles de costo revueltos entre ciudades (N15). **Los
β_η son sensibles a los controles de costo:** sustituir solo la fila de la Ciudad de México
(de la mediana de las demás ciudades a su total estatal del censo) mueve Pan de 1.233 a 1.373
y Frutas de 0.590 a 0.752, sin cambiar qué sectores son significativos. Es evidencia directa
a favor de D-H. Pollo y
huevo es frágil: cruza el umbral de significancia según se incluya o no Pan de caja (t 3.52
sin la categoría, 2.17 con ella).

## 4.2 Bienestar — dirección sólida, magnitud revisada

*Cifras de `comparacion_2014_2022.ipynb`: ambos años sin recorte, denominador `ing_cor`, y
con las cuatro correcciones de fidelidad al Gauss en la ruta de bienestar (V1–V4, §5.1).*

| decil | 2014 | 2022 | cambio |
|---|---|---|---|
| 1 (más pobre) | 8.6 % | **11.1 %** | +2.5 |
| 5 | 3.6 % | 6.7 % | +3.1 |
| 10 (más rico) | 1.0 % | 2.4 % | +1.4 |
| **Total** | **3.2 %** | **5.9 %** | **+2.7** |
| **Regresividad D1/D10** | **8.90** | **4.61** | **−4.3** |
| Reducción del Gini | 2.0 % | 2.9 % | +0.9 |

**Dos resultados, y el segundo es el más interesante:**

1. **La carga del poder de mercado casi se duplicó** (3.2 % → 5.9 % del ingreso corriente,
   +84 %). *Ojo: 2022 incluye un sector significativo más (pan de caja) que 2014 no mide;
   parte del aumento es cobertura, no cambio económico.*
2. **Y se volvió notablemente menos regresiva.** En 2014 el decil más pobre soportaba una
   carga 8.9 veces la del más rico; en 2022, 4.6 veces.

El patrón por decil lo explica: el aumento en **puntos porcentuales** es parejo (+1.4 a
+3.1), pero como el decil 1 partía de una base mucho más alta, en términos **relativos**
crece menos abajo que en medio.

**Lectura de política:** el problema creció en tamaño y se volvió menos concentrado en los
hogares más pobres. No es *"el impuesto se duplicó y sigue castigando igual a los pobres"* —
ésa fue la lectura preliminar de este trabajo, antes de corregir la ruta de bienestar.

> **Sobre la validación de la ruta corregida:** con V1–V4 y el denominador `ing_cor` en
> ambos años, la réplica 2014 daba una reducción del Gini de 7.3 %, idéntica a la publicada.
> Pero **ésa no es la combinación del Gauss**: el original calcula el Gini sobre `ing_mon`
> (l.938) y la VE sobre `ing_total` (l.6519). Con la combinación correcta la réplica da
> **5.7 %** contra el 7.3 % publicado, y el Gini observado sigue coincidiendo exacto (0.481).
> La coincidencia previa era artefacto de usar una sola variable de ingreso para las dos
> cosas.

---

# 5. Limitaciones

### 5.1 La ruta de bienestar: cuatro divergencias corregidas

La primera versión de este trabajo reportaba una variación equivalente implausible —hasta
el 77 % del gasto en categorías— y una carga que se duplicaba entre años. Al localizar la
implementación de referencia en el Gauss (l.6380–6519) aparecieron **cuatro divergencias**,
todas en la ruta de bienestar:

| # | el Gauss | nuestra versión previa | ubicación |
|---|---|---|---|
| V1 | umbral de significancia `cdfni(0.99)` = **t ≥ 2.326** | t ≥ 1.645 (95 %) | l.6376 |
| V2 | markup del contrafactual = **−1/ε** (índice de Lerner) | `p/cm` de la regresión NEIO | l.6345, 6380 |
| V3 | denominador de VE/ingreso = **ingreso corriente** | ingreso monetario | l.6519 |
| V4 | se reportan **medianas** | medias | l.6520 |

**V2 y V4 interactúan**: evaluadas por separado dan una lectura invertida —V2 parecía
empeorar la réplica— y solo al aplicarlas juntas se ve que ambas la mejoran. Con las cuatro,
la réplica 2014 reproduce la reducción del Gini publicada de forma exacta (7.3 %).

**Nota metodológica para el white paper.** Las cuatro divergencias estaban en la sección de
bienestar, **la parte del pipeline que menos se auditó porque sus resultados parecían
razonables**. Las secciones que producían números absurdos —elasticidades pegadas a −1,
sobreprecios de 400 %— se revisaron a fondo desde el inicio. La plausibilidad protegió al
error durante todo el proyecto.

**Advertencia sobre `ing_total` vs `ing_cor`.** El Gauss divide por la columna 22
(`ing_total`); las cifras de §4.2 usan `ing_cor` (columna 23), que difiere ~4 % y es la
**única variable de ingreso con definición idéntica en 2014 y 2022** — la ENIGH 2022
"Nueva serie" ya no publica `ing_total` ni `ing_mon`. Para la comparación entre años no hay
alternativa; para la réplica pura de 2014 debería usarse `ing_total`.

**Estado de los notebooks por año.** Los tres notebooks de resultados usan V1–V4 y se
re-ejecutaron el 15 de septiembre de 2026 (0 errores). Los notebooks por año reportan cada
año en su mejor configuración (2014 con recorte, denominador `ing_total`); las cifras
comparativas citadas aquí vienen de `comparacion_2014_2022.ipynb`.

### 5.2 Categorías que no deben reportarse como cambio económico

* **Transporte foráneo** — su valor de 2014 en la comparación (0.169) proviene de la
  configuración sin recorte, que degrada precisamente esta categoría (con recorte: 0.800).
* **Carne de res** — se mueve 1.0 puntos entre años; es la categoría peor identificada en
  ambos, por D-E (las vísceras pesan 7× de más).
* **Bebidas** — mayor desviación respecto de 2014; candidata a estar afectada por el 34 % de
  hogares que conserva efectos ingreso débilmente identificados incluso sin recorte.

### 5.3 Comparabilidad entre años

* Los Gini de 2014 (0.481, sobre `ing_mon`) y 2022 (0.446) **no son comparables**: la ENIGH
  2022 "Nueva serie" dejó de publicar el ingreso monetario y hubo que reconstruirlo. La
  comparación de §4 usa `ing_cor`, única variable con definición idéntica en ambos años.
* La ENIGH 2022 tiene 90,102 hogares contra 19,124 en 2014. Las tasas de retención tras
  filtros son casi iguales (64.4 % y 65.8 %), pero los errores estándar de 2022 son
  mecánicamente menores.

### 5.4 Datos

* **Área Metropolitana de la Ciudad de México** — el mercado más grande — no tiene serie de
  INPC propia (usa el factor mediano nacional) y no aparece en el extracto municipal de los
  Censos Económicos 2023 (sus controles de costo son la mediana de las demás ciudades).
* Materiales de construcción: 4 ciudades sin serie de INPP propia (mediana).
* Dos genéricos de precio (pan de caja y nutricionales) venían truncados en la descarga
  original para 22–23 ciudades; se recuperaron de INEGI y se validaron contra una ciudad de
  control (15 de septiembre de 2026).

---

# 6. Agenda abierta

**Ordenada por relación entre esfuerzo y valor para el white paper.**

### Alta prioridad

1. ✅ **Pan industrial como sector propio** (15 sep 2026): β_η = 0.971, t = 10.01 (§2, D-F).
   Pendiente derivado: medirlo también en 2014 con `precios_pan_de_caja_2014_46_ciudades.asc`
   del CD.

2. ✅ **Evaluación de la variación equivalente** corregida (V1–V4, §5.1).

3. **Cuantificar el sesgo de D-H.** Con el resultado de D-F es aún más urgente: el sector
   atomizado (panadería) es el de mayor β_η medido. Construir controles de costo por sector desde los Censos
   Económicos y reestimar. Permitiría decir *cuánto* del `β_η` publicado es poder de mercado
   y cuánto es variación de costos no controlada.

### Media

4. Resolver la brecha muestral de 15,586 hogares (D-C).
5. Verificar si Bebidas y Transporte reflejan cambio económico o residuo de identificación.
6. Serie de INPC propia para el Área Metropolitana de la CDMX.

### Extensión

7. **ENIGH 2024.** La arquitectura ya lo contempla: escribir `datos_2024.cargar()` que
   devuelva un `DatosAnio`. El núcleo de cálculo no se toca.

---

# Anexo — Reproducibilidad

Todo lo citado se regenera con los notebooks del repositorio. Desde el directorio que
contiene `Replica_COFECE/`:

| notebook | produce | tiempo |
|---|---|---|
| `aradillas_2014.ipynb` | réplica 2014 (§1) | ~30 s |
| `aradillas_2022.ipynb` | actualización 2022 | ~6 min |
| `comparacion_2014_2022.ipynb` | comparación bajo tratamiento idéntico (§4) | ~7 min |

Los datos no están versionados: son microdatos públicos de INEGI. Sí lo está `CD/`, con el
programa Gauss original y los `.asc` preprocesados — **material irreemplazable**, obtenido
por solicitud de acceso a la información.

La celda siguiente verifica que el entorno esté completo y lista las referencias de línea
del Gauss usadas en §2, para quien quiera comprobarlas.

In [1]:
import os, sys
sys.path.insert(0, "Replica_COFECE/Codigo")

for m in ["aradillas_core", "datos_base", "datos_2014", "datos_2022"]:
    print(f"  {'ok ' if os.path.exists(f'Replica_COFECE/Codigo/{m}.py') else 'FALTA'} {m}.py")
gauss = "Replica_COFECE/CD/programa_ENIGH_2014.g"
print(f"  {'ok ' if os.path.exists(gauss) else 'FALTA'} {gauss}")

print("\nReferencias de línea en programa_ENIGH_2014.g (§2):")
for d, l in [("D-A  estimación en una sola etapa OLS", "2452-5753"),
             ("D-B  filtro de vivienda propia", "1101"),
             ("D-D  recorte dentro del bucle", "5671-5722"),
             ("D-E  piso numérico sobre gastos nulos", "1766 y análogas"),
             ("D-F  composición de la categoría Pan", "1979-1990"),
             ("D-H  controles de costo sin dimensión sectorial", "6182-6209")]:
    print(f"  {d:<48} l.{l}")

  ok  aradillas_core.py
  ok  datos_base.py
  ok  datos_2014.py
  ok  datos_2022.py
  ok  Replica_COFECE/CD/programa_ENIGH_2014.g

Referencias de línea en programa_ENIGH_2014.g (§2):
  D-A  estimación en una sola etapa OLS            l.2452-5753
  D-B  filtro de vivienda propia                   l.1101
  D-D  recorte dentro del bucle                    l.5671-5722
  D-E  piso numérico sobre gastos nulos            l.1766 y análogas
  D-F  composición de la categoría Pan             l.1979-1990
  D-H  controles de costo sin dimensión sectorial  l.6182-6209
